# Quaternions: Understanding q = −q

Quaternions are a number system that extends complex numbers. They are used extensively in 3D graphics, robotics, and aerospace to represent **rotations without gimbal lock**.

A quaternion is written as:

$$q = w + xi + yj + zk$$

where:
- `w` is the **scalar (real) part**
- `x, y, z` are the **vector (imaginary) parts**
- `i, j, k` satisfy: $i^2 = j^2 = k^2 = ijk = -1$

---
### What we'll cover:
1. Building a `Quaternion` class from scratch
2. Encoding 3D rotations as unit quaternions
3. Proving that **q and −q represent the same rotation**
4. Visualising the double-cover property

## Step 1 — Imports

We only need NumPy (for maths) and Matplotlib (for plotting). No special quaternion library — we build everything ourselves so every step is transparent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Make plots look clean
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'
plt.rcParams['font.family'] = 'sans-serif'

print('Imports OK ✓')

## Step 2 — Build the Quaternion Class

We define a `Quaternion` class that stores `(w, x, y, z)` and implements:
- **Multiplication** — the key non-commutative operation
- **Conjugate** — flip sign of vector part
- **Norm** — length of the quaternion
- **Normalise** — project onto the unit 4-sphere (required for rotations)
- **Negation** — simply negate all four components

> Quaternion multiplication is **not commutative**: `p*q ≠ q*p` in general.

In [ ]:
class Quaternion:
    """
    A unit quaternion q = w + xi + yj + zk
    stored as a NumPy array [w, x, y, z].
    """

    def __init__(self, w, x, y, z):
        self.q = np.array([w, x, y, z], dtype=float)

    # ── properties ──────────────────────────────────────────────
    @property
    def w(self): return self.q[0]

    @property
    def x(self): return self.q[1]

    @property
    def y(self): return self.q[2]

    @property
    def z(self): return self.q[3]

    @property
    def vec(self):
        """The vector (imaginary) part as a 3D array."""
        return self.q[1:]

    # ── norm & normalise ─────────────────────────────────────────
    def norm(self):
        """Euclidean length: sqrt(w² + x² + y² + z²)"""
        return np.linalg.norm(self.q)

    def normalise(self):
        """Return a unit quaternion (norm == 1)."""
        return Quaternion(*(self.q / self.norm()))

    # ── conjugate ───────────────────────────────────────────────
    def conjugate(self):
        """q* = w - xi - yj - zk  (flip vector sign)"""
        return Quaternion(self.w, -self.x, -self.y, -self.z)

    # ── negation ─────────────────────────────────────────────────
    def __neg__(self):
        """−q = −w − xi − yj − zk  (negate ALL components)"""
        return Quaternion(*(-self.q))

    # ── multiplication ───────────────────────────────────────────
    def __mul__(self, other):
        """
        Hamilton product.  Not commutative!
        Derived from the rules i²=j²=k²=ijk=−1.
        """
        w1, x1, y1, z1 = self.q
        w2, x2, y2, z2 = other.q
        return Quaternion(
            w1*w2 - x1*x2 - y1*y2 - z1*z2,   # w
            w1*x2 + x1*w2 + y1*z2 - z1*y2,   # i
            w1*y2 - x1*z2 + y1*w2 + z1*x2,   # j
            w1*z2 + x1*y2 - y1*x2 + z1*w2    # k
        )

    # ── display ──────────────────────────────────────────────────
    def __repr__(self):
        w, x, y, z = self.q
        def fmt(v, label):
            sign = '+' if v >= 0 else ''
            return f'{sign}{v:.4f}{label}'
        return f'Q({self.w:.4f} {fmt(x,"i")} {fmt(y,"j")} {fmt(z,"k")})'


# Quick sanity check
q = Quaternion(1, 0, 0, 0)   # identity
print('Identity quaternion:', q)
print('Norm:', q.norm())

## Step 3 — Create a Rotation Quaternion

To rotate by angle **θ** around a unit axis **(ax, ay, az)**, the rotation quaternion is:

$$q = \cos\!\left(\frac{\theta}{2}\right) + \sin\!\left(\frac{\theta}{2}\right)\,(a_x\,i + a_y\,j + a_z\,k)$$

The **half-angle** is the key quirk: a 360° rotation in 3D needs a 720° journey through quaternion space.

In [ ]:
def rotation_quaternion(axis, angle_deg):
    """
    Build a unit quaternion that rotates `angle_deg` degrees
    around `axis` (a 3-element array-like).

    Steps:
      1. Normalise the axis so it has length 1.
      2. Convert angle to radians and halve it.
      3. Apply the rotation quaternion formula.
    """
    # 1. Ensure the rotation axis is a unit vector
    axis = np.array(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)

    # 2. Half-angle in radians
    half = np.deg2rad(angle_deg) / 2.0

    # 3. Quaternion components
    w = np.cos(half)
    x, y, z = np.sin(half) * axis

    return Quaternion(w, x, y, z)


# Example: 90° rotation around the Z axis
q90 = rotation_quaternion([0, 0, 1], angle_deg=90)
print(f'q (90° around Z):  {q90}')
print(f'norm:              {q90.norm():.6f}   ← must be 1.0')
print()

# Its negation — this is what we want to prove equals the same rotation
neg_q90 = -q90
print(f'−q (90° around Z): {neg_q90}')
print(f'norm:              {neg_q90.norm():.6f}   ← also 1.0')

## Step 4 — Apply a Rotation to a 3D Vector

To rotate a 3D vector **v** by quaternion **q**, we use the **sandwich product**:

$$v' = q \cdot \tilde{v} \cdot q^*$$

where $\tilde{v}$ is the vector embedded as a pure quaternion `(0, vx, vy, vz)`, and `q*` is the conjugate of q.

The result's vector part is the rotated vector.

In [ ]:
def rotate_vector(q, v):
    """
    Rotate 3D vector v by unit quaternion q.

    Method: v' = q * pure(v) * q.conjugate()
    """
    # Embed the 3D vector as a pure quaternion (w=0)
    v_pure = Quaternion(0, v[0], v[1], v[2])

    # Sandwich product
    rotated = q * v_pure * q.conjugate()

    # The rotated vector lives in the imaginary (vector) part
    return rotated.vec


# Test: rotate the X-axis unit vector by 90° around Z
# Expected result: [0, 1, 0]  (X → Y)
v_original = np.array([1.0, 0.0, 0.0])

v_by_q   = rotate_vector( q90, v_original)
v_by_neg = rotate_vector(-q90, v_original)

print('Original vector:         ', v_original)
print('Rotated by  q:           ', np.round(v_by_q,   6))
print('Rotated by −q:           ', np.round(v_by_neg, 6))
print()
print('Are they the same?', np.allclose(v_by_q, v_by_neg))

## Step 5 — Prove q = −q Algebraically

Let's verify this for **many different rotations**, not just one, to make the result concrete.

In [ ]:
# Test across a sweep of angles and random axes
rng = np.random.default_rng(42)
angles = np.linspace(0, 360, 37)       # every 10°
axes   = rng.normal(size=(37, 3))      # 37 random rotation axes

test_vec = np.array([1.0, 2.0, 3.0])   # arbitrary vector to rotate

all_same = True
print(f"{'Angle':>8}  {'Axis':>24}  {'q == -q?':>10}")
print('-' * 55)

for angle, axis in zip(angles, axes):
    q   = rotation_quaternion(axis, angle)
    nq  = -q

    v1 = rotate_vector( q, test_vec)
    v2 = rotate_vector(nq, test_vec)

    same = np.allclose(v1, v2, atol=1e-10)
    if not same:
        all_same = False

    ax_unit = axis / np.linalg.norm(axis)
    print(f"{angle:>7.1f}°  [{ax_unit[0]:+.2f},{ax_unit[1]:+.2f},{ax_unit[2]:+.2f}]  {'✓ same' if same else '✗ DIFF'}")

print()
print('All rotations identical for q and −q:', all_same)

## Step 6 — The Rotation Matrix Perspective

We can convert a quaternion to a 3×3 rotation matrix. If q and −q represent the same rotation, their matrices must be **identical**.

In [ ]:
def quaternion_to_matrix(q):
    """
    Convert unit quaternion to a 3x3 rotation matrix.

    The formula expands v' = q * pure(v) * q* directly:
      R[0,0] = 1 - 2(y² + z²),  R[0,1] = 2(xy - wz),  etc.
    """
    w, x, y, z = q.w, q.x, q.y, q.z
    return np.array([
        [1 - 2*(y*y + z*z),   2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),       1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y),       2*(y*z + w*x),     1 - 2*(x*x + y*y)]
    ])


# Pick an interesting rotation
q_test = rotation_quaternion([1, 1, 0], angle_deg=120)

R_q   = quaternion_to_matrix( q_test)
R_neg = quaternion_to_matrix(-q_test)

print('Rotation matrix from  q:')
print(np.round(R_q,   4))
print()
print('Rotation matrix from −q:')
print(np.round(R_neg, 4))
print()
print('Matrices are identical:', np.allclose(R_q, R_neg))

## Step 7 — Visualise the Double Cover

The **double cover** means that as θ goes from 0° → 360°, the quaternion travels through one hemisphere of S³ (the 4-sphere). To return to the start, θ must go all the way to **720°** — passing through the antipodal hemisphere (−q).

We plot:
- The **w component** of q vs angle — it traces `cos(θ/2)`
- The corresponding **3D rotation angle** — it traces `θ`
- Highlighting where q switches to −q territory

In [ ]:
theta = np.linspace(0, 720, 1000)   # 0 → 720° (two full rotations in 3D)
half  = np.deg2rad(theta) / 2       # half-angle

w_vals = np.cos(half)               # scalar part of q

fig, axes_plot = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
fig.suptitle('The Double Cover: q and −q on S³', fontsize=14, fontweight='bold', y=1.01)

# ── Top plot: w component ────────────────────────────────────────
ax1 = axes_plot[0]

# q hemisphere (θ: 0 → 360, w goes +1 → -1)
mask_q  = theta <= 360
mask_nq = theta >= 360

ax1.plot(theta[mask_q],  w_vals[mask_q],  color='#EF9F27', lw=2.5, label='q  (first cover)')
ax1.plot(theta[mask_nq], w_vals[mask_nq], color='#378ADD', lw=2.5, label='−q (second cover)')
ax1.axhline(0,   color='gray', lw=0.7, ls='--')
ax1.axvline(360, color='gray', lw=0.7, ls='--')
ax1.fill_between(theta[mask_q],  w_vals[mask_q],  alpha=0.08, color='#EF9F27')
ax1.fill_between(theta[mask_nq], w_vals[mask_nq], alpha=0.08, color='#378ADD')
ax1.set_ylabel('w = cos(θ/2)', fontsize=11)
ax1.set_ylim(-1.2, 1.2)
ax1.legend(loc='upper right')
ax1.set_title('Scalar part of quaternion across 720° journey', fontsize=11)
ax1.annotate('q starts here\n(w = +1)', xy=(0, 1), xytext=(30, 1.05),
             fontsize=9, color='#EF9F27',
             arrowprops=dict(arrowstyle='->', color='#EF9F27', lw=1.2))
ax1.annotate('−q hemisphere\nbegins (w = −1)', xy=(360, -1), xytext=(390, -0.6),
             fontsize=9, color='#378ADD',
             arrowprops=dict(arrowstyle='->', color='#378ADD', lw=1.2))
ax1.annotate('back to start\n(w = +1)', xy=(720, 1), xytext=(620, 1.05),
             fontsize=9, color='#378ADD',
             arrowprops=dict(arrowstyle='->', color='#378ADD', lw=1.2))

# ── Bottom plot: 3D rotation angle ──────────────────────────────
ax2 = axes_plot[1]
ax2.plot(theta[mask_q],  theta[mask_q],  color='#EF9F27', lw=2.5)
ax2.plot(theta[mask_nq], theta[mask_nq], color='#378ADD', lw=2.5)
ax2.axvline(360, color='gray', lw=0.7, ls='--')
ax2.set_xlabel('Quaternion journey angle (degrees)', fontsize=11)
ax2.set_ylabel('3D rotation angle (degrees)', fontsize=11)
ax2.set_title('3D rotation angle is the same for q and −q at matched points', fontsize=11)

# Mark the q / -q equivalence: 90° is encoded by BOTH 90° and 450°
for pair in [(90, 450), (180, 540)]:
    a, b = pair
    ax2.annotate('', xy=(b, b), xytext=(a, a),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=1.2))
    ax2.text((a+b)/2, (a+b)/2 + 18, f'{a%360}° rotation\nsame by q & −q',
             ha='center', fontsize=8, color='purple')

for ax in axes_plot:
    ax.set_xticks([0, 90, 180, 270, 360, 450, 540, 630, 720])
    ax.set_xticklabels(['0°','90°','180°','270°','360°\n(q→−q)','450°','540°','630°','720°'])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8 — Visualise the Rotated Vector in 3D

Here we show **q and −q both rotating the same vector to the same destination** across a range of angles. The arrow colours match: they land on top of each other exactly.

In [ ]:
def draw_arrow(ax, origin, direction, color, lw=2, label=None):
    """Draw a 3D arrow on the given axes."""
    ax.quiver(*origin, *direction,
              color=color, linewidth=lw,
              arrow_length_ratio=0.15,
              label=label)


# Sample some rotation angles to plot side by side
sample_angles = [30, 90, 150, 210]
axis = np.array([0, 0, 1])           # rotate around Z
v0   = np.array([1.0, 0.0, 0.0])    # start at the X axis

fig = plt.figure(figsize=(14, 4))
fig.suptitle('q and −q rotating the same vector — result is always identical',
             fontsize=13, fontweight='bold')

for idx, angle in enumerate(sample_angles):
    ax = fig.add_subplot(1, 4, idx+1, projection='3d')

    q   = rotation_quaternion(axis, angle)
    nq  = -q

    v_q   = rotate_vector( q, v0)
    v_nq  = rotate_vector(nq, v0)

    O = [0, 0, 0]

    # Original vector (gray)
    draw_arrow(ax, O, v0, 'gray', lw=1.5, label='original')

    # Rotated by q (amber, thick)
    draw_arrow(ax, O, v_q,  '#EF9F27', lw=3, label='q rotates to')

    # Rotated by −q (blue, dashed) — offset very slightly so both are visible
    # In practice they overlap perfectly; we nudge for visibility
    draw_arrow(ax, O, v_nq + np.array([0, 0, 0.05]), '#378ADD', lw=2, label='−q rotates to')

    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_zlim(-0.3, 0.3)
    ax.set_xlabel('X', fontsize=8)
    ax.set_ylabel('Y', fontsize=8)
    ax.set_title(f'{angle}° rotation', fontsize=10)
    ax.tick_params(labelsize=6)
    ax.set_box_aspect([1,1,0.3])
    ax.view_init(elev=25, azim=30)

    if idx == 0:
        ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()

print('Note: amber (q) and blue (−q) arrows land on the same spot.')
print('The blue arrow is offset by 0.05 in Z only to make both visible.')

## Step 9 — Animate the Full 720° Journey

This animation sweeps θ from 0° → 720°. Watch the quaternion marker travel both hemispheres of S³ while the 3D rotation repeats identically in the second half.

> **Run this cell** to generate the animation (may take a few seconds to render).

In [ ]:
import matplotlib.patches as mpatches

fig2, (ax_q, ax_3d) = plt.subplots(1, 2, figsize=(12, 5))
fig2.suptitle('720° journey through quaternion space (double cover)', fontsize=13)

# ── Left: w-component track ──────────────────────────────────────
# Pre-compute the full track so we can draw static background curves
all_theta = np.linspace(0, 720, 400)
all_half  = np.deg2rad(all_theta) / 2
all_w     = np.cos(all_half)

ax_q.plot(all_theta[:200], all_w[:200], color='#EF9F27', lw=1.5, alpha=0.4)
ax_q.plot(all_theta[200:], all_w[200:], color='#378ADD', lw=1.5, alpha=0.4)
ax_q.axhline(0,   color='gray', lw=0.5, ls='--')
ax_q.axvline(360, color='gray', lw=0.5, ls='--', label='q → −q switch')
ax_q.set_xlim(0, 720)
ax_q.set_ylim(-1.3, 1.3)
ax_q.set_xlabel('θ (degrees)', fontsize=10)
ax_q.set_ylabel('w = cos(θ/2)', fontsize=10)
ax_q.set_title('Scalar part (w)', fontsize=11)
ax_q.legend(fontsize=9)
ax_q.grid(True, alpha=0.3)

# Animated dot and label that ride along the track
dot_q,  = ax_q.plot([], [], 'o', markersize=9, color='#EF9F27', zorder=5)
label_q = ax_q.text(0, 0, '', fontsize=9, ha='left')

# ── Right: rotating arrow in 2D (top-down XY view) ──────────────
# We use ax.quiver() instead of annotate() because quiver objects
# expose set_UVC() and set_color() which work safely during animation.
ax_3d.set_xlim(-1.4, 1.4)
ax_3d.set_ylim(-1.4, 1.4)
ax_3d.set_aspect('equal')
ax_3d.set_xlabel('X', fontsize=10)
ax_3d.set_ylabel('Y', fontsize=10)
ax_3d.set_title('3D rotation (top-down view)', fontsize=11)
circle = plt.Circle((0, 0), 1, fill=False, color='lightgray', lw=0.8)
ax_3d.add_patch(circle)
ax_3d.axhline(0, color='lightgray', lw=0.5)
ax_3d.axvline(0, color='lightgray', lw=0.5)
ax_3d.grid(True, alpha=0.2)

# quiver: origin at (0,0), initial direction along X
# scale=1 + scale_units='xy' keeps the arrow exactly 1 unit long
arrow_q = ax_3d.quiver(
    0, 0,          # tail position
    1, 0,          # (U, V) direction components
    angles='xy', scale_units='xy', scale=1,
    color='#EF9F27', width=0.012, zorder=5
)

angle_text = ax_3d.text(0, -1.25, '', ha='center', fontsize=10)
phase_text = ax_3d.text(0,  1.25, '', ha='center', fontsize=9, color='gray')

def init():
    dot_q.set_data([], [])
    label_q.set_text('')
    angle_text.set_text('')
    phase_text.set_text('')
    return dot_q, label_q, angle_text, phase_text

def update(frame):
    theta_val = frame * (720 / 119)        # 120 frames → 0° to 720°
    half_val  = np.deg2rad(theta_val) / 2
    w_val     = np.cos(half_val)

    # Which cover are we in?
    phase = 'q (first cover)'  if theta_val < 360 else '−q (second cover)'
    color = '#EF9F27'           if theta_val < 360 else '#378ADD'

    # ── w-track dot ──────────────────────────────────────────────
    dot_q.set_data([theta_val], [w_val])
    dot_q.set_color(color)
    label_q.set_position((min(theta_val + 8, 690), w_val))
    label_q.set_text(f'w={w_val:.2f}')
    label_q.set_color(color)

    # ── rotating arrow ───────────────────────────────────────────
    # Compute the new (U, V) direction from the 3D rotation angle
    rot_rad = np.deg2rad(theta_val % 360)
    u, v    = np.cos(rot_rad), np.sin(rot_rad)

    # set_UVC updates direction; set_color updates colour — both are
    # proper quiver methods that work during FuncAnimation saving.
    arrow_q.set_UVC(u, v)
    arrow_q.set_color(color)

    # ── text labels ──────────────────────────────────────────────
    angle_text.set_text(
        f'3D angle: {theta_val % 360:.0f}°   |   journey: {theta_val:.0f}°'
    )
    phase_text.set_text(phase)
    phase_text.set_color(color)

    return dot_q, label_q, angle_text, phase_text

anim = FuncAnimation(
    fig2, update, frames=120,
    init_func=init, interval=60,
    blit=False, repeat=True
)

plt.tight_layout()
HTML(anim.to_jshtml())

## Step 10 — Key Takeaways

Here's a concise summary of everything proved above:

In [ ]:
print('=' * 60)
print('  QUATERNION DOUBLE COVER — SUMMARY')
print('=' * 60)

points = [
    ('q = w + xi + yj + zk',
     'Four numbers encode a 3D rotation.'),

    ('q = cos(θ/2) + sin(θ/2)·axis',
     'Uses HALF the rotation angle — the key quirk.'),

    ('v\' = q · v_pure · q*',
     'Sandwich product applies the rotation to any vector.'),

    ('q and −q → same rotation matrix',
     'Negating all four components changes nothing in 3D.'),

    ('720° quaternion journey = 360° in 3D',
     'S³ is a double cover of SO(3) — the double cover property.'),

    ('Why it matters',
     'Smooth interpolation (SLERP), no gimbal lock, compact storage.'),
]

for title, detail in points:
    print(f'\n  ▸ {title}')
    print(f'    {detail}')

print()
print('=' * 60)

# Final numerical proof
q_final  = rotation_quaternion([1, 1, 1], angle_deg=137)
nq_final = -q_final
v_test   = np.array([3.0, -1.0, 2.0])

r1 = rotate_vector( q_final, v_test)
r2 = rotate_vector(nq_final, v_test)

print(f'\nFinal proof — 137° around [1,1,1], vector {v_test}:')
print(f'  q  = {q_final}')
print(f'  −q = {nq_final}')
print(f'  rotate by  q: {np.round(r1, 6)}')
print(f'  rotate by −q: {np.round(r2, 6)}')
print(f'  Identical: {np.allclose(r1, r2)}  ✓')

---
### Further reading
- **3Blue1Brown** — *Quaternions and 3D rotation* (YouTube)
- **Wikipedia** — *Quaternions and spatial rotation*
- **Kuipers, J.B.** — *Quaternions and Rotation Sequences* (Princeton University Press)